# DL200 · Notebook 01 — Chain-of-Thought Summary
## MLP for Regression: Key Code Terms & Intuition

> Read this notebook as a mental model walk-through, not as runnable code.
> Each step shows **what** the code does → **why** it matters → **the key term** to remember.

---

## Step 1 — Reproducibility & Hardware

```python
set_global_seed(42)   # fix all RNGs (numpy, torch, python random)
device = get_device() # 'cuda' if GPU available, else 'cpu'
```

| Key term | Intuition |
|---|---|
| `seed` | Locks the random number generator so every run produces identical results |
| `device` | Where tensors live; GPU = massive parallelism for matrix ops |

**Chain of thought:** _"Before anything, pin randomness and pick the compute device."_

---

## Step 2 — Generate Synthetic Data

```python
X, y = make_regression(n_samples=1000, n_features=1, noise=15.0)
# True model: y = X·w + ε,  ε ~ N(0, 15²)
```

| Key term | Intuition |
|---|---|
| `make_regression` | Manufactures a dataset where we **know** the ground-truth relationship |
| `noise=15.0` | Adds Gaussian scatter — models real-world measurement error |
| `true_coef` | The actual slope; we can verify the model learned it |

**Chain of thought:** _"Synthetic data lets us test our pipeline on a problem with a known answer."_

---

## Step 3 — Split & Scale

```python
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

scaler_X = StandardScaler()
X_train_scaled = scaler_X.fit_transform(X_train)  # fit ON TRAIN ONLY
X_test_scaled  = scaler_X.transform(X_test)        # transform only

scaler_y = StandardScaler()
y_train_scaled = scaler_y.fit_transform(y_train.reshape(-1,1)).flatten()
```

| Key term | Intuition |
|---|---|
| `train_test_split` | Hold-out set to measure generalisation, never seen during training |
| `StandardScaler` | Shifts mean to 0, std to 1 → gradients are balanced, training is stable |
| `fit_transform` vs `transform` | **Fit on train only** — fitting on test would leak future info (data leakage) |
| Scale **y** too | MSE on raw targets can dwarf MSE on scaled ones; keeps loss values interpretable |

**Chain of thought:** _"Split first, scale after — and scale y not just X for stable regression loss."_

---

## Step 4 — Wrap in PyTorch DataLoader

```python
train_ds = TensorDataset(
    torch.tensor(X_train_scaled, dtype=torch.float32),
    torch.tensor(y_train_scaled, dtype=torch.float32),
)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
```

| Key term | Intuition |
|---|---|
| `TensorDataset` | Zips (X, y) tensor pairs so they stay aligned when shuffled |
| `DataLoader` | Iterator that feeds **mini-batches** to the model |
| `batch_size=64` | Mini-batch SGD: noisy but fast; full-batch would be slow & memory-heavy |
| `shuffle=True` | Randomise order each epoch — prevents the model learning data order |

**Chain of thought:** _"DataLoader = the conveyor belt feeding mini-batches to the training loop."_

---

## Step 5 — Build the MLP Architecture

```
Input(1) → Linear(64) → ReLU → Linear(32) → ReLU → Linear(16) → ReLU → Linear(1)
```

```python
class RegressionMLP(nn.Module):
    def __init__(self, input_dim, hidden_dims=(64, 32, 16)):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for h_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, h_dim))
            layers.append(nn.ReLU())
            prev_dim = h_dim
        layers.append(nn.Linear(prev_dim, 1))   # ← no activation!
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x).squeeze(-1)       # (batch,1) → (batch,)
```

| Key term | Intuition |
|---|---|
| `nn.Module` | Base class; PyTorch tracks all parameters automatically |
| `nn.Linear(in, out)` | Learnable weight matrix + bias: `y = xW + b` |
| `nn.ReLU()` | `max(0, x)` — introduces non-linearity, lets the net learn curves |
| `nn.Sequential` | Chains layers so `forward` is just one call |
| **No activation on output** | Regression needs unbounded real values; sigmoid/softmax would cage predictions in [0,1] |
| `.squeeze(-1)` | Removes trailing size-1 dim so shape matches loss function expectation: `(batch,)` not `(batch,1)` |

**Chain of thought:** _"Stack Linear+ReLU for hidden layers; naked Linear at the end for free-range regression output."_

---

## Step 6 — The Training Loop

```python
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(epochs):
    # --- TRAIN ---
    model.train()                          # 1. enable dropout/batchnorm train mode
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()              # 2. wipe old gradients
        preds = model(X_batch)             # 3. forward pass
        loss  = loss_fn(preds, y_batch)    # 4. compute loss
        loss.backward()                    # 5. backprop → fill .grad
        optimizer.step()                   # 6. nudge weights

    # --- VALIDATE ---
    model.eval()                           # 7. disable dropout
    with torch.no_grad():                  # 8. no gradient tracking needed
        ...
```

| Key term | Intuition |
|---|---|
| `Adam` | Adaptive learning rate per parameter; works well out of the box |
| `model.train()` | Activates layers like Dropout that behave differently during training |
| `optimizer.zero_grad()` | **Critical:** gradients accumulate by default — reset before each batch |
| `loss.backward()` | Autograd walks the computation graph backward, fills `.grad` on every parameter |
| `optimizer.step()` | Uses `.grad` values to update weights: `w ← w − lr·∇w` |
| `model.eval()` | Switches to inference mode — no dropout noise |
| `torch.no_grad()` | Skips gradient tracking → faster, less memory during validation |

**Chain of thought:** _"Each batch: zero → forward → loss → backward → step. Each eval: eval mode + no_grad."_

---

## Step 7 — Evaluation Metrics

```python
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

mse  = mean_squared_error(y_true, y_pred)   # average squared error
mae  = mean_absolute_error(y_true, y_pred)  # average absolute error
r2   = r2_score(y_true, y_pred)             # 1 = perfect, 0 = mean-baseline, <0 = worse than mean
```

| Key term | Intuition |
|---|---|
| `MSE` | Penalises large errors heavily (quadratic); unit is squared |
| `MAE` | Penalises errors linearly; same unit as target |
| `R²` | Fraction of variance explained; 0.5 = model explains 50 % of variation |
| Residual plot | If residuals are random around 0, the model has no systematic bias |
| Predicted vs Actual | Points hugging the diagonal line = good fit |

**Chain of thought:** _"MSE for training loss, R² for human-readable quality, residual plot for bias diagnosis."_

---

## Step 8 — Loss Functions vs Outliers

```python
nn.MSELoss()         # L2: (y - ŷ)²          — sensitive to outliers
nn.L1Loss()          # L1: |y - ŷ|            — robust, but gradient discontinuity at 0
nn.HuberLoss(delta=1.0)  # quadratic when |e|≤δ, linear otherwise — best of both
```

| Loss | Outlier effect | Gradient behaviour |
|---|---|---|
| MSE | Outlier error gets **squared** → dominates gradient update | Smooth, grows with error |
| MAE | Outlier gets same weight as normal point | Constant ±1 (kink at 0) |
| Huber | Outlier handled linearly, small errors handled quadratically | Smooth everywhere |

**Experiment result:**  
- MSE train loss stayed ~5× higher than clean data (outliers inflating it)  
- Huber val MSE was **lowest** — most robust to the 30 injected outliers  

**Chain of thought:** _"If data has outliers, Huber loss is the pragmatic default. MSE is fine for clean data."_

---

## Step 9 — Common Mistakes Cheat-Sheet

| Mistake | Symptom | Fix |
|---|---|---|
| Forget `zero_grad()` | Gradients accumulate → erratic loss | Call before each `backward()` |
| Forget `model.train()` / `model.eval()` | Dropout active during eval | Always toggle |
| Activation on output layer | Predictions stuck in [0,1] | No activation for regression |
| Not scaling y | Huge loss values, unstable training | `StandardScaler` on both X and y |
| Fit scaler on test set | Optimistic test metrics (data leakage) | `fit_transform` train, `transform` test |
| No `.squeeze(-1)` | Shape mismatch `(batch,1)` vs `(batch,)` in loss | Add `.squeeze(-1)` in `forward` |
| LR too high | Loss explodes or oscillates | Start with `1e-3`, halve if unstable |
| LR too low | Loss barely moves | Increase or use a scheduler |

---

## Full Chain-of-Thought at a Glance

```
seed + device
    │
    ▼
make_regression  →  raw (X, y)
    │
    ▼
train_test_split  →  (X_train, X_test, y_train, y_test)
    │
    ▼
StandardScaler (fit on train only)  →  scaled tensors
    │
    ▼
TensorDataset  →  DataLoader (batch_size=64, shuffle=True)
    │
    ▼
RegressionMLP (nn.Module)
  Linear → ReLU → Linear → ReLU → Linear → ReLU → Linear  [no final activation]
    │
    ▼
Training Loop (per epoch, per batch):
  model.train()
  zero_grad() → forward → loss_fn → backward() → step()
  model.eval() + no_grad() → validation loss
    │
    ▼
Evaluation: MSE / MAE / R²  +  predicted-vs-actual + residual plot
    │
    ▼
Loss comparison with outliers:
  MSE (sensitive) vs MAE (robust) vs Huber (best of both)  → Huber wins
```

---

**Next:** [02 — MLP for Classification](02_MLP_for_Classification_MNIST_or_SklearnDigits.ipynb)